# broadcast-initial-weights — faded example 3: Build the flattened-state fingerprint for a sync assertion

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-initial-weights`. Running the beacon reports progress on the `Distributed: broadcast initial weights` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: broadcast initial weights` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-initial-weights`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-initial-weights"
DD_SUBTOPIC = "Distributed: broadcast initial weights"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A common DDP test flattens every state tensor into one vector and checks equality against rank 0. The fingerprint must concatenate every `state_dict()` tensor reshaped to 1-D, in the deterministic state-dict order, so vectors from different ranks line up element-for-element.

## Faded exercise 3

### Complete the fingerprint helper

`broadcast_state` is already correct. Complete `flat_state` so it returns a single 1-D float tensor that concatenates every tensor in the model's state dict. The post-broadcast comparison below should then report `True` for all ranks.

**Fill in:** Concatenates every state_dict tensor reshaped to 1-D and cast to float into one vector.

In [ ]:
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Linear(2, 2, bias=True)
    with t.no_grad():
        m.weight.fill_(float(rank + 1))
        m.bias.fill_(float(-(rank + 1)))
    return m

def broadcast_state(model, fake):
    fake.reset()
    for tensor in model.state_dict().values():
        fake.broadcast(tensor, src=0)

def flat_state(model):
    vec = None  # TODO: concatenate every state_dict tensor reshaped to 1-D and cast to float
    return vec

rank0 = build_model(0)
ref = flat_state(rank0)
fake = FakeDist([v.clone() for v in rank0.state_dict().values()])
m = build_model(1)
broadcast_state(m, fake)
print(t.equal(flat_state(m), ref))


def _test():
    r0 = build_model(0)
    ref = flat_state(r0)
    assert ref.ndim == 1, 'fingerprint must be 1-D'
    expected_len = sum(v.numel() for v in r0.state_dict().values())
    assert ref.numel() == expected_len, 'must include every state element'
    f = FakeDist([v.clone() for v in r0.state_dict().values()])
    m1 = build_model(1)
    assert not t.equal(flat_state(m1), ref), 'precondition: ranks differ before broadcast'
    broadcast_state(m1, f)
    assert t.equal(flat_state(m1), ref), 'fingerprints must match rank0 after broadcast'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Linear(2, 2, bias=True)
    with t.no_grad():
        m.weight.fill_(float(rank + 1))
        m.bias.fill_(float(-(rank + 1)))
    return m

def broadcast_state(model, fake):
    fake.reset()
    for tensor in model.state_dict().values():
        fake.broadcast(tensor, src=0)

def flat_state(model):
    vec = t.cat([v.reshape(-1).float() for v in model.state_dict().values()])
    return vec

rank0 = build_model(0)
ref = flat_state(rank0)
fake = FakeDist([v.clone() for v in rank0.state_dict().values()])
m = build_model(1)
broadcast_state(m, fake)
print(t.equal(flat_state(m), ref))
```
</details>